# Virtualize a few example datasets

1. Grab URLS from CEDA test catalog
2. Virtualize them to OSN bucket
3. Verify that snippet to download them works

In [10]:
import httpx
import icechunk as ic
from cmip7_virtualization.virtualize import virtualize_from_urls
from cmip7_virtualization.storage import osn_storage, http_vccs_from_registry
from cmip7_virtualization.catalog import urls_from_stac_item
from cmip7_virtualization.store import repo_exists

In [11]:
BUCKET = 'leap-pangeo-pipeline'
ROOT_PREFIX = 'cmip7-virtualization'
CEDA_STAC  = "https://api.stac.esgf.ceda.ac.uk"
COLLECTION = "CMIP6"
N_ITEMS = 4

In [12]:
import subprocess

def op_read(ref):
    return subprocess.check_output(["op", "read", ref]).decode().strip()

key    = op_read("op://Work/z6baienaiyhiexztlbbonbeaka/Read-Write/Access_Key")
secret = op_read("op://Work/z6baienaiyhiexztlbbonbeaka/Read-Write/Secret_Access_Key")

In [13]:
r = httpx.get(f"{CEDA_STAC}/collections/{COLLECTION}/items?limit={N_ITEMS}", timeout=30)
r.raise_for_status()
all_items = r.json()["features"]

In [17]:
for item in all_items:
    
    cmip_id = item.get('id', False)
    assert cmip_id
    prefix = f"{ROOT_PREFIX}/{cmip_id}/"
    storage = osn_storage(
        bucket=BUCKET,
        prefix=prefix,
        access_key_id=key,
        secret_access_key=secret
        )
    
    # skip if repo already exists: (for now skip, since the ref building is slow, we can later think about updates?)
    if repo_exists(storage):
        print(f"skipping {prefix=} — already exists")
        continue    
    
    print(f"Building references and committing to icechunk \n {prefix=}")

    # Build references
    urls = urls_from_stac_item(item)
    vds, registry = virtualize_from_urls(urls)
    vccs = http_vccs_from_registry(registry=registry)
    config = ic.RepositoryConfig.default()
    for vcc in vccs:
        config.set_virtual_chunk_container(vcc)
    repo = ic.Repository.open_or_create(
        storage=storage,
        config=config
    )
    session = repo.writable_session('main')
    vds.vz.to_icechunk(session.store)
    snapshot_id = session.commit("cmip")
    repo.save_config()
    print(snapshot_id)

skipping prefix='cmip7-virtualization/CMIP6.VolMIP.NERC.UKESM1-0-LL.volc-pinatubo-full.r9i1p1f2.day.zg.gn.v20230810/' — already exists
skipping prefix='cmip7-virtualization/CMIP6.VolMIP.NERC.UKESM1-0-LL.volc-pinatubo-full.r9i1p1f2.day.wap.gn.v20230810/' — already exists
skipping prefix='cmip7-virtualization/CMIP6.VolMIP.NERC.UKESM1-0-LL.volc-pinatubo-full.r9i1p1f2.day.uas.gn.v20230810/' — already exists
skipping prefix='cmip7-virtualization/CMIP6.VolMIP.NERC.UKESM1-0-LL.volc-pinatubo-full.r9i1p1f2.day.ta.gn.v20230810/' — already exists


# snippet to download the data locally
You can download the icechunk stores locally using the AWS CLI

```
aws s3 cp s3://leap-pangeo-pipeline/cmip7-virtualization/<prefix>/ \
  ./local-icechunk/ \
  --recursive \
  --no-sign-request \
  --endpoint-url https://nyu1.osn.mghpcc.org
```
with any of the above prefixes, for example:

```
aws s3 cp s3://leap-pangeo-pipeline/cmip7-virtualization/CMIP6.VolMIP.NERC.UKESM1-0-LL.volc-pinatubo-full.r9i1p1f2.day.zg.gn.v20230810/ \
  ./local-icechunk/ \
  --recursive \
  --no-sign-request \
  --endpoint-url https://nyu1.osn.mghpcc.org
```